# ByteFrost — Price Prediction Prototype (XGBoost)

**Owner:** Agni Pratap Pramanik (AI/ML)  
**Sprint 1:** Aug 26–31, 2026  
**Status:** Demo MUST-HAVE — complete

This notebook trains an XGBoost regressor on mandi price data to predict a recommended price band for a produce listing. It uses synthetic data while Aradhya's real mandi price research is in progress.

In [ ]:
# macOS: XGBoost needs OpenMP runtime
import os
os.environ['DYLD_LIBRARY_PATH'] = '/opt/homebrew/opt/libomp/lib:' + os.environ.get('DYLD_LIBRARY_PATH', '')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

print('Imports OK')

## 1. Load synthetic mandi price data

In [ ]:
df = pd.read_csv('../data/synthetic_mandi_prices.csv')
df['date'] = pd.to_datetime(df['date'])
print(f'Rows: {len(df):,}')
print(f'Crops: {df["crop"].nunique()}, Mandis: {df["mandi"].nunique()}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.head()

## 2. Feature engineering

In [ ]:
FEATURE_COLS = [
    'crop_encoded', 'mandi_encoded', 'month', 'week_of_year', 'day_of_year',
    'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_std_7',
    'quantity_log', 'quantity_lag_7',
]

def engineer_features(df):
    df = df.copy()
    df['month'] = df['date'].dt.month
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['day_of_year'] = df['date'].dt.dayofyear
    df = df.sort_values(['crop', 'mandi', 'date']).reset_index(drop=True)
    g = df.groupby(['crop', 'mandi'])['price_per_quintal']
    df['lag_7'] = g.shift(7)
    df['lag_14'] = g.shift(14)
    df['lag_30'] = g.shift(30)
    df['rolling_mean_7'] = g.transform(lambda x: x.rolling(7, min_periods=1).mean())
    df['rolling_std_7'] = g.transform(lambda x: x.rolling(7, min_periods=1).std())
    df['quantity_log'] = np.log1p(df['quantity'])
    df['quantity_lag_7'] = df.groupby(['crop', 'mandi'])['quantity_log'].shift(7)
    df['crop_encoded'] = df['crop'].astype('category').cat.codes
    df['mandi_encoded'] = df['mandi'].astype('category').cat.codes
    return df

df = engineer_features(df)
df = df.dropna(subset=['lag_7', 'lag_14', 'lag_30', 'quantity_lag_7']).reset_index(drop=True)
print(f'Rows after dropping rows without lag features: {len(df):,}')
df[['date', 'crop', 'mandi', 'price_per_quintal'] + FEATURE_COLS].head()

## 3. Time-based train/test split

In [ ]:
max_date = df['date'].max()
cutoff = max_date - pd.DateOffset(months=3)
train = df[df['date'] < cutoff]
test = df[df['date'] >= cutoff]
print(f'Train: {len(train):,} rows ({train["date"].min().date()} to {train["date"].max().date()})')
print(f'Test:  {len(test):,} rows ({test["date"].min().date()} to {test["date"].max().date()})')

X_train, y_train = train[FEATURE_COLS], train['price_per_quintal']
X_test, y_test = test[FEATURE_COLS], test['price_per_quintal']

## 4. Train XGBoost

In [ ]:
model = XGBRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('Training complete')

## 5. Evaluate

In [ ]:
preds = model.predict(X_test)
rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
print(f'RMSE: {rmse:.2f} Rs/quintal')
print(f'MAE:  {mae:.2f} Rs/quintal')
print(f'MAPE: {mape:.2f}%')
print(f'MAE as % of mean price: {mae / y_test.mean() * 100:.2f}%')

## 6. Feature importance

In [ ]:
imp = sorted(zip(FEATURE_COLS, model.feature_importances_), key=lambda x: x[1], reverse=True)
plt.figure(figsize=(8, 5))
sns.barplot(x=[i[1] for i in imp[:10]], y=[i[0] for i in imp[:10]])
plt.title('XGBoost Feature Importance (Price Prediction)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 7. Save model for serving

In [ ]:
import joblib, json, os
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/price_prediction_xgb.joblib')
crop_map = dict(enumerate(df['crop'].astype('category').cat.categories))
mandi_map = dict(enumerate(df['mandi'].astype('category').cat.categories))
meta = {
    'feature_cols': FEATURE_COLS,
    'crop_map': {str(k): v for k, v in crop_map.items()},
    'mandi_map': {str(k): v for k, v in mandi_map.items()},
    'metrics': {'rmse': float(rmse), 'mae': float(mae), 'mape': float(mape)},
    'trained_on': 'synthetic_mandi_prices.csv',
}
with open('../models/price_prediction_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Model + metadata saved to ../models/')